# 08 · Errors & Debugging

Every programmer writes bugs, every day. What makes someone good is how quickly they **find** them. Today you learn to read error messages properly, handle errors on purpose, and debug like a scientist.

**By the end of this lesson you can:**
- Tell syntax errors, runtime errors and logic errors apart
- Read a stack trace in JavaScript and a traceback in Python
- Throw and catch errors (`throw` / `try` / `catch`, `raise` / `try` / `except`)
- Debug systematically: reproduce, hypothesise, test, fix

In [ ]:
# ▶️ Run this cell first, every time you open the notebook.
# It lets %%node cells run JavaScript and gives you check() in both languages.
%load_ext course_helpers

## 🔁 Warm-up
Answer from memory first: **no scrolling back, no notes**. Getting it wrong is fine. Trying to remember is what makes it stick.

1. `const a = [1]; const b = a; b.push(2);` What is `a` now?
2. What does `[3, 1, 2].map((n) => n * 2)` return?
3. Why does `[10, 9, 1].sort()` give a strange order in JavaScript?
4. What does `git switch -c name` do?

**My answers** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️
4. ✏️

## Part 1: Three kinds of errors

| Kind | When | Example | How you notice |
|---|---|---|---|
| **Syntax error** | before the program starts | a missing `)` or quote | an error, and nothing runs |
| **Runtime error** | while the program runs | using something that is `undefined` | an error, and the program stops at that line |
| **Logic error** | never "crashes" | `<` instead of `<=` | the answer is **wrong** |

Logic errors are the hardest, because the computer doesn't complain. That's why we test (with `check`).

## Part 2: Reading an error message

In [ ]:
%%node
function getInitial(user) {
  return user.name[0].toUpperCase();
}

function greet(user) {
  return `Hi, ${getInitial(user)}!`;
}

console.log(greet({ name: "ada" }));
console.log(greet({ nickname: "linus" }));

### Anatomy of an error message
```
cell:2                                        ← where: line 2 of the cell
  return user.name[0].toUpperCase();
                  ^                           ← the exact spot

TypeError: Cannot read properties of undefined (reading '0')
└─ type ─┘ └──────────────── message ──────────────────────┘
    at getInitial (cell:2:19)                 ← where it crashed (the most recent call)
    at greet (cell:6:17)                      ← who called getInitial
    at cell:10:13                             ← who called greet
```

The list of `at ...` lines is the **stack trace**: the call stack from lesson 05 at the moment of the crash.
In JavaScript, read it **from the top**: where it crashed, then who called that, and so on.

🔍 Answer these questions (double-click to edit):

1. Which line crashed?
2. Which value was `undefined`, and why?
3. Which line of *our* code passed in the bad data?

Here's the same bug in Python:

In [ ]:
def get_initial(user):
    return user["name"][0].upper()

def greet(user):
    return f"Hi, {get_initial(user)}!"

print(greet({"name": "ada"}))
print(greet({"nickname": "linus"}))

🔍 Python prints its **traceback** in the **opposite** order: "most recent call **last**". So read
Python tracebacks **from the bottom up**: the last line is the error, and just above it is where it
happened. Same information, different order.

| Situation | JavaScript | Python |
|---|---|---|
| the name doesn't exist | `ReferenceError` | `NameError` |
| wrong type of value, or using `undefined` / `None` | `TypeError` | `TypeError` / `AttributeError` |
| the code isn't valid | `SyntaxError` | `SyntaxError` / `IndentationError` |
| the index or key doesn't exist | (no error: `undefined`) | `IndexError` / `KeyError` |
| a bad value | `RangeError` (sometimes) | `ValueError` |

## Part 3: Handling errors with `try` / `catch`
Some errors are expected: the user typed letters instead of a number, or the network is down.
You can **catch** them, so the program doesn't crash.

In [ ]:
%%node
function parseAge(text) {
  const age = Number(text);
  if (Number.isNaN(age) || age < 0) {
    throw new Error(`"${text}" is not a valid age`);
  }
  return age;
}

for (const input of ["21", "abc", "-4", "35"]) {
  try {
    const age = parseAge(input);
    console.log("✅ age:", age);
  } catch (err) {
    console.log("⚠️ problem:", err.message);
  }
}
console.log("The program keeps running!");

In [ ]:
def parse_age(text):
    age = int(text)    # raises ValueError for "abc"
    if age < 0:
        raise ValueError(f'"{text}" is not a valid age')
    return age

for text in ["21", "abc", "-4", "35"]:
    try:
        age = parse_age(text)
        print("✅ age:", age)
    except ValueError as err:
        print("⚠️ problem:", err)
print("The program keeps running!")

### 🔍 Investigate
- `throw` (JavaScript) and `raise` (Python) create an error on purpose: "I can't continue with this input."
- `try { ... } catch (err) { ... }` / `try: ... except ValueError as err:`: if anything inside `try`
  fails, jump to `catch` / `except`.
- Python makes you name the **type** of error you catch. That's a good habit: only catch errors you know
  how to handle. Catching *everything* hides bugs.
- There's also `finally`, which runs whether there was an error or not (used for cleaning up).

In [ ]:
%%node
// 🔮 My prediction:
console.log(10 / 0);
console.log(-10 / 0);
console.log(0 / 0);

In [ ]:
# 🔮 My prediction:
print(10 / 0)

Once again JavaScript doesn't complain (`Infinity`, `NaN`), while Python stops with a
`ZeroDivisionError`.

## Part 4: A debugging strategy
Debugging is the scientific method:
1. **Reproduce**: find an input that makes the bug happen every time.
2. **Read** the error message, all of it (if there is one).
3. **Make a hypothesis**: "I think `total` is wrong after the loop."
4. **Test it**: print the values you're unsure about, e.g. `console.log("total is", total)`.
   Compare what you *think* the value is with what it *really* is.
5. **Change one thing**, then run again.
6. Still stuck? **Explain the code line by line, out loud**, to a person or a rubber duck 🦆.
   You'll often hear the bug yourself.

Here is a bug. `average([10, 20, 30])` should be 20:

In [ ]:
%%node
function average(numbers) {
  let total = 0;
  for (let i = 1; i < numbers.length; i++) {
    total += numbers[i];
  }
  return total / numbers.length;
}

console.log(average([10, 20, 30]));

Let's test the hypothesis "the loop doesn't add what I think" with a print inside the loop:

In [ ]:
%%node
function average(numbers) {
  let total = 0;
  for (let i = 1; i < numbers.length; i++) {
    console.log(`i=${i}, adding ${numbers[i]}, total before: ${total}`);
    total += numbers[i];
  }
  return total / numbers.length;
}

console.log(average([10, 20, 30]));

🔍 The print shows that `i` starts at 1, so the first number (10) is never added. The fix: `let i = 0`.
Remember to remove your debug prints afterwards.

> 💡 **Debuggers.** In VS Code you can click to the left of a line number to set a **breakpoint**, then
> step through the code line by line and watch every variable change. It's an automatic trace table!
> For Python cells, choose **Debug Cell** from the menu next to the cell's ▶ button. For JavaScript,
> you'll use it with real `.js` files (Run and Debug, F5) in the capstone project.

## 🐛 Bug hunt
Each program below has **one bug** (or maybe two 😉). Reproduce, hypothesise, test, fix. When every check
shows ✅, describe the bug in the comment on the first line.

### Bug 1

In [ ]:
%%node
// 🐛 The bug was:
function countPassing(grades) {
  let count = 0;
  for (const g of grades) {
    if (g > 60) {
      count++;
    }
  }
  return count;
}

check(countPassing([50, 60, 70]), 2);   // 60 counts as passing
check(countPassing([90, 91]), 2);

### Bug 2

In [ ]:
%%node
// 🐛 The bug was:
function findMax(numbers) {
  let max = 0;
  for (const n of numbers) {
    if (n > max) {
      max = n;
    }
  }
  return max;
}

check(findMax([3, 8, 2]), 8);
check(findMax([-5, -2, -9]), -2);

### Bug 3

In [ ]:
%%node
// 🐛 The bug was:
function containsNegative(numbers) {
  for (const n of numbers) {
    if (n < 0) {
      return true;
    } else {
      return false;
    }
  }
}

check(containsNegative([-1, 2, 3]), true);
check(containsNegative([1, 2, -3]), true);
check(containsNegative([1, 2, 3]), false);
check(containsNegative([]), false);

### Bug 4

In [ ]:
%%node
// 🐛 The bug was:
function sortedCopy(numbers) {
  return numbers.sort();
}

const data = [10, 2, 33, 4];
check(sortedCopy(data), [2, 4, 10, 33]);
check(data, [10, 2, 33, 4]);   // the original must not change

### Bug 5 (Python)

In [ ]:
# 🐛 The bug was:
def total_price(items):
    total = 0
    for item in items:
        total = item["price"] * item["quantity"]
    return total

cart = [{"price": 10, "quantity": 2}, {"price": 5, "quantity": 4}]
check(total_price(cart), 40)
check(total_price([]), 0)

### Bug 6 (Python)

In [ ]:
# 🐛 The bug was:
def initials(full_name):
    parts = full_name.split(" ")
    result = ""
    for i in range(1, len(parts)):
        result += parts[i][0].upper()
    return result

check(initials("ada lovelace"), "AL")
check(initials("grace brewster murray hopper"), "GBMH")

## 🌍 Rosetta Stone: same idea, different language
The ideas are the same in every language. Only the spelling changes. Add your own rows when you discover something new.

| Idea | JavaScript | Python |
|---|---|---|
| Create an error | `throw new Error("msg")` | `raise ValueError("msg")` |
| Catch errors | `try { } catch (err) { }` | `try:` / `except ValueError as err:` |
| Always run | `finally { }` | `finally:` |
| The error's message | `err.message` | `str(err)` |
| Order of the stack trace | most recent call **first** | most recent call **last** |
| Divide by zero | `Infinity` / `NaN` | `ZeroDivisionError` |
| Is it NaN? | `Number.isNaN(x)` | `math.isnan(x)` |

## ✍️ Explain it back
Answer in your own words, as if explaining it to a friend who missed today's session.

1. Name the three kinds of errors, and how you notice each one.
2. Describe your debugging strategy in four steps.
3. Which bug in the bug hunt was the hardest to find, and why?

**My explanations** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️

## 🌱 Git checkpoint
A **merge conflict** happens when two branches change **the same line** and Git can't know which
version you want. It's normal, not a disaster. Practise on a small text file (not a notebook, since
notebooks are messy JSON inside):

1. Create a file `favourite.txt` in VS Code with one line: `My favourite language is JavaScript`. Save it.
   ```bash
   git add favourite.txt
   git commit -m "Add favourite language"
   git switch -c python-fan
   ```
2. Change the line to `My favourite language is Python`. Save.
   ```bash
   git commit -am "Make Python my favourite"   # -a = add all changed files
   git switch main
   ```
3. The file says JavaScript again. Change the line to `My favourite language is Rust`. Save.
   ```bash
   git commit -am "Make Rust my favourite"
   git merge python-fan                         # 💥 CONFLICT
   ```
4. Open `favourite.txt`:
   ```
   <<<<<<< HEAD
   My favourite language is Rust
   =======
   My favourite language is Python
   >>>>>>> python-fan
   ```
   Above `=======` is your current branch's version (`HEAD`), below it is the other branch's.
   Edit the file to say what you want and **delete the marker lines** (VS Code also has buttons:
   *Accept Current / Incoming / Both*). Then:
   ```bash
   git add favourite.txt
   git commit -m "Merge python-fan and resolve the conflict"
   git log --oneline --graph
   git push
   ```

Don't forget to commit today's notebook too.